# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a complete, step-by-step guide for loading, inspecting, and processing the FAIR² dataset using the `mlcroissant` Python library. The FAIR² dataset includes mass spectrometry data on protein abundance, modifications, and sequences from human mast cells' extracellular vesicles.

### Dataset Source
The dataset is defined by a [Croissant schema](https://mlcommons.github.io/croissant/) hosted at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

- Use this URL to dynamically access and explore the dataset.


In [ ]:
# Ensure mlcroissant is installed (uncomment if needed)
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and basic records via `mlcroissant`. This step retrieves the dataset structure—record sets, fields, and context—directly from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

pd.set_option('display.max_columns', 100)

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print a short summary using the metadata attributes
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Dataset ID: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")


## 2. Data Overview
Inspect the available record sets and their fields, using the `@id` values for precise referencing and further extraction.

In [ ]:
# List available record sets and their fields using @id references
print("Record Sets in the dataset:")
record_sets = []
if hasattr(metadata, "recordSet"):
    # Some datasets may provide a single dict, others a list
    rs = metadata.recordSet
    if isinstance(rs, dict):
        rs = [rs]
    for recset in rs:
        # Each record set has an @id and a list of fields
        recset_id = getattr(recset, "@id", None) or recset.get("@id")
        name = getattr(recset, "name", None) or recset.get("name", "")
        record_sets.append(recset_id)
        print(f"  - RecordSet @id: {recset_id} | Name: {name}")
        # Fields
        fields = getattr(recset, "field", None) or recset.get("field", [])
        if isinstance(fields, dict):
            fields = [fields]
        print("    Fields:")
        for f in fields:
            fid = getattr(f, "@id", None) or f.get("@id")
            fname = getattr(f, "name", None) or f.get("name", "")
            print(f"      - Field @id: {fid} | Name: {fname}")
else:
    print("  No record sets found in the metadata.")

### Preview some records
Below we preview several records (entries/rows) for a known `recordSet` by its `@id`.
- Replace `<record_set_id>` below with an actual @id (see previous cell).


In [ ]:
# List and preview up to 2 record sets (by @id)
# We'll pick the first available record set if present
if record_sets:
    sample_record_set = record_sets[0]
    print(f"Preview records from RecordSet @id: {sample_record_set}")
    for i, rec in enumerate(dataset.records(record_set=sample_record_set)):
        print(f"Record #{i+1}:", rec)
        if i >= 4:
            break
else:
    print("No record sets available for preview.")

## 3. Data Extraction
We now extract all data from one or more record sets into pandas DataFrames to facilitate processing and analysis.

- All operations below reference record sets and fields by their `@id`.

In [ ]:
# Create a DataFrame for each record set
dataframes = {}
for rsid in record_sets:
    print(f"Extracting RecordSet @id: {rsid}")
    # Not all record sets might have records; handle exceptions
    try:
        records = list(dataset.records(record_set=rsid))
        if len(records):
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error extracting records: {e}")

# Show columns for the main record set (if at least one exists)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print("Column names in DataFrame:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

Select a numeric field using its `@id`, filter records, normalize, and group by a category.

- Below, replace `<numeric_field_id>` and `<group_field_id>` as appropriate, using columns present in the loaded DataFrame, which are based on field `@id`s.

Typical numeric fields: mass (`MW`), peptide counts, coverage percentage, abundances, etc.

In [ ]:
# Choose main record set for analysis
if dataframes:
    df = dataframes[main_record_set_id]
    print('Sample columns:', df.columns.tolist())
    # Guess a numeric field (try common candidates)
    possible_numeric = [col for col in df.columns if any(s in col.lower() for s in ['mw', 'abundance', 'count', 'coverage', 'intensity'])]
    print('Possible numeric fields:', possible_numeric)
    if possible_numeric:
        numeric_field_id = possible_numeric[0]
        # Ensure numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > average ({threshold:.2f}): {len(filtered_df)} records")
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print("\nSample of normalized field:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Guess a group-by field (e.g., 'description', 'sample', 'protein', etc.)
        possible_group = [col for col in df.columns if any(g in col.lower() for g in ['desc', 'protein', 'sample', 'accession', 'group'])]
        if possible_group:
            group_field_id = possible_group[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id} (showing up to 5 groups):")
            display(grouped_df.head())
        else:
            print('No suitable group field found for grouping.')
    else:
        print('No numeric field found for EDA.')
else:
    print('No DataFrame loaded for EDA.')

## 5. Visualization

Use matplotlib to visualize numeric field distributions or relationships (e.g., field histograms, boxplots, or group comparisons).

In [ ]:
# Plot histogram of the numeric field and a boxplot by group
if dataframes and possible_numeric:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If group field identified, visualize mean by group (if not too many groups)
    if 'group_field_id' in locals() and group_field_id in df.columns and df[group_field_id].nunique() < 30:
        plt.figure(figsize=(10,4))
        df.groupby(group_field_id)[numeric_field_id].mean().sort_values().plot(kind='bar')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} By {group_field_id}")
        plt.show()
else:
    print('Visualization unavailable (no numeric data loaded).')

## 6. Conclusion

In this notebook, you explored the structure and contents of the FAIR² protein dataset using `mlcroissant`, referencing all dataset entities by their Croissant `@id`. We loaded metadata and record sets, extracted tabular data, performed basic filtering and normalization on a numeric field, grouped results by a categorical attribute, and visualized distributions.

This workflow demonstrates how FAIR data sources can be programmatically explored and prepared for downstream bioinformatics or data science analysis.
